# Testing Notebook for EdGB Fisher Analysis

## Goal

This notebook tests the PhenomXHM waveform with a -1PN EdGB phase correction. The model used below is `PhenomXHM_EdGB()`, whose extra Fisher parameter is `alpha_EdGB_km_2 = alpha_EdGB^2` in km$^4$. The relative inclusion in the original GWJulia can be found in the $\texttt{.jl}$ file with the same name. 

## Current code organization

* `PhenomXHM_EdGB.jl` contains the EdGB-specific mapping from `alpha_EdGB_km_2` to the generic TIGER/ppE coefficient `delta_phi_minus2` (for a -1PN addition). The expression for the correction is given by: 

$$\delta \phi_{-2} = \frac{128.0}{3.0} \ \beta_\mathrm{EdGB} \ \eta^{(-2 / 5)}$$
 
where $\beta_\mathrm{EdGB}$ is the expression found in Eq. (4) of [arXiv:1905.00870v3](https://arxiv.org/abs/1905.00870v3).

* `PhenomXHM_TIGER_spinless(-1.0)` (-1.0 indicates the correction) is available as the generic model where the extra parameter is directly `delta_phi_minus2` and is the one called in the methods to return the polarizations hphc. 

* `PhenomXHM.jl` and `ConnectionFunctionsXAS.jl` only carry the generic -1PN phase plumbing, through the `PhenomXHM_TIGER_spinless(-1.0)` structure.


In [1]:
using GWInference
using LinearAlgebra
using DSP
using Statistics

[ Info: Precompiling GWInference [12bfb5b4-b11b-482e-a53a-61106a071c3c](cache misses: include_dependency fsize change (1))
[ Info: Precompiling GWInference [12bfb5b4-b11b-482e-a53a-61106a071c3c] (cache misses: include_dependency fsize change (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [ ]:
# GW170608-like, strongest/cleanest EdGB test (from paper at least)
m1 = 11.0
m2 = 7.6
mc = (m1*m2)^(3/5) / (m1 + m2)^(1/5)
eta = m1*m2 / (m1 + m2)^2

chi_eff = 0.03
chi1 = chi_eff * (m1 + m2) / m1
chi2 = 0.0

dL = 0.32  # Gpc
squared_alpha_EdGB_km4 = 2.0^4 # from reference paper cited treshold


theta = 1.0
phi = 2.0
iota = 0.7
psi = 0.4
tcoal = 0.0
phiCoal = 0.0

0.0

## Test 1

Test whether the waveform with $o1=0.0$ is the same as the one without passing the BGR factor.

In [29]:
f = collect(range(10.0, 512.0, length=2000))

# std GR model
hp_gr, hc_gr = hphc(PhenomXHM(), f, mc, eta, chi1, chi2, dL, iota)

# BGR model with sqrt_alpha_EdGB_km = 0.0 --> means zero added effect in o1 parameter
hp_0, hc_0 = hphc(
    PhenomXHM_EdGB(),
    f, mc, eta, chi1, chi2, dL, iota, 0.0
)

println("GR vs BGR(alpha_EdGB_km^2=0): ", maximum(abs.(hp_gr .- hp_0)))

GR vs BGR(alpha_EdGB_km^2=0): 0.0


## Test 2

Test whether the waveform with a nonzero EdGB coupling is different from GR and finite. The diagnostic value is intentionally not tiny, otherwise the difference is below floating-point precision.

In [30]:
# BGR model with sqrt_alpha_EdGB_km = 0.3

hp_bgr, hc_bgr = hphc(
    PhenomXHM_EdGB(),
    f, mc, eta, chi1, chi2, dL, iota, 0.3
)

println("GR vs BGR(alpha_EdGB_km^2=0.3): ", maximum(abs.(hp_gr .- hp_bgr)))
println("finite? ", all(isfinite, real.(hp_bgr)), " ", all(isfinite, imag.(hp_bgr)))

GR vs BGR(alpha_EdGB_km^2=0.3): 1.8396817050127563e-23
finite? true true


## Test 3

Check the expected EdGB scaling. Since the parameter is simple the square of `alpha_EdGB_km`, namely `alpha_EdGB_km_2`, the phase correction scales approximately as `alpha_EdGB_km_2^2`, so doubling the parameter should give a ratio close to 2 for small values:

In [31]:
hp_a, _ = hphc(PhenomXHM_EdGB(), f, mc, eta, chi1, chi2, dL, iota, 0.3)
hp_b, _ = hphc(PhenomXHM_EdGB(), f, mc, eta, chi1, chi2, dL, iota, 0.6)

dh_a = hp_a .- hp_gr
dh_b = hp_b .- hp_gr

println("quadratic scaling ratio: ", norm(dh_b) / norm(dh_a))

quadratic scaling ratio: 1.9294893996142366


## Test 4: Fisher analysis

Checks Fisher on waveforms other than mine, and on mine. Then calculate the SNR and covariance to extract errors. 

In [ ]:
FisherMatrix(
    PhenomD(),
    CE1Id,
    mc, eta, chi1, chi2, dL,
    theta, phi, iota, psi, tcoal, phiCoal;
    res=100,
    fmin=10.0,
    fmax=512.0,
    rho_thres=nothing
);

println("Fisher matrix calculated")

11×11 Matrix{Float64}:
     1.65022e9      -1.31609e9  …      -1.7156e9         5.82258e6
    -1.31609e9       1.10045e9          1.57112e9       -4.42643e6
    -3.40188e8       2.82072e8          3.96592e8       -1.15363e6
    -2.11449e8       1.74878e8          2.4344e8   -718312.0
 -7165.13       -17059.3              -34.0699          -1.56497e-13
     1.2803e7       -4.94648e6  …       1.7041e7     71457.9
     7.04058e5      -3.11753e6         -1.61696e7    -8290.57
 26127.4        -24341.0           -26413.9            108.425
     1.12393e7      -8.54425e6         -1.07718e7    44227.6
    -1.7156e9        1.57112e9          3.27394e9       -5.58039e6
     5.82258e6      -4.42643e6  …      -5.58039e6    22912.5

In [ ]:
model = PhenomXHM_EdGB()

# find the Fisher matrix for the BGR model with sqrt_alpha_EdGB_km = 0.3

F= FisherMatrix(
    model,
    ETS,
    mc,
    eta,
    chi1,
    chi2,
    dL,
    theta,
    phi,
    iota,
    psi,
    tcoal,
    phiCoal,
    alpha_EdGB_km_2;
    res=300,
    fmin=10.0,
    fmax=512.0,
    rho_thres=nothing,
    #return_SNR=true
);

println("Fisher matrix calculated")

In [ ]:
println(size(F))
println("is symmetric? (0 means true) = ", maximum(abs.(F .- F')) )
println("is finite? = ", all(isfinite, F))
println("diag = ", diag(F))

(12, 12)
symmetric = 0.0
finite = true
diag = [6.47310846188415e9, 7.607693374818768e11, 3.014614027483931e8, 1.1402253239121445e8, 688856.5170939952, 5.761925503405852e6, 3.647627620246202e6, 36844.688492655165, 278219.8878316736, 2.5900268117816856e10, 70538.90735042504, 221597.07962456334]


In [ ]:
snr = SNR(
    model,
    ETS,
    mc, eta, chi1, chi2, dL,
    theta, phi, iota, psi, tcoal,
    alpha_EdGB_km_2;
    res=100,
    fmin=10.0,
    fmax=512.0
)

println("signal SNR = ", snr)

SNR = 264.87156416532616


In [ ]:
mycovariance = CovMatrix(F);

Inversion successful
Inversion error: 3.2379587623789218e-6



In [39]:
# we can now calculate the errors on the parameters

myerrors = Errors(mycovariance)
parameters_string = ["mc", "η", "χ_1", "χ_2", "dL", "θ", "ϕ", "ι", "ψ", "tcoal", "Φ_coal", "α_EdGB^2 [km^4]"]

for i in eachindex(myerrors)
    println("The error on $(parameters_string[i]) is $(myerrors[i])")
end

The error on mc is 0.000769644005015637
The error on η is 0.0029038704855011313
The error on χ_1 is 0.100346808223362
The error on χ_2 is 0.14883828320659054
The error on dL is 0.04257038494166193
The error on θ is 0.022756739178830365
The error on ϕ is 0.015209601579506246
The error on ι is 0.12366257568446415
The error on ψ is 0.9157240172304264
The error on tcoal is 0.0005029375946329842
The error on Φ_coal is 1.8210648263222111
The error on α_EdGB^2 [km^4] is 5.827633260090358


In [ ]:
# find errors on alpha_EdGB, which is the param in the paper we are interested in:

sigma_alpha_squared = myerrors[end]
sigma_sqrt_alpha = sigma_alpha_squared^(1.0 / 4.0)

println("The error on sqrt(α_EdGB^2) is $(sigma_sqrt_alpha)")

The error on sqrt(α_EdGB^2) is 1.5537210631321803


It's important to notice that I am studying the Fisher matrix for the squared value of the parameters alpha in a way that the derivative with respect to such parameter will be linear and not singular for small values of such parameter. 